In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image, ImageDraw
from datasets import load_dataset
from huggingface_hub import login

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
HF_REPO  = "dfreddi/multimodal-deid"
SPLIT    = "base"
REVIEW_PATH = Path("output/review") / f"{SPLIT}_review.json"
REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
token = os.environ.get("HF_TOKEN")
if token:
    login(token=token)
else:
    print("HF_TOKEN not set — will only work for public repos")

ds = load_dataset(HF_REPO, split=SPLIT)
print(f"Loaded {len(ds)} samples from '{HF_REPO}' (split: {SPLIT})")
print("Columns:", ds.column_names)

In [ ]:
_PARENT_COLORS = {
    "NAME":          "#e74c3c",
    "DATE_OF_BIRTH": "#e67e22",
    "DATE":          "#ff9800",
    "AGE":           "#f1c40f",
    "ID":            "#9b59b6",
    "CONTACT":       "#e91e63",
    "ADDRESS":       "#3498db",
}

_label_color: dict[str, str] = {}
rejected: set[int] = set()

def _color_for(label: str) -> str:
    if label not in _label_color:
        parent = label.split(":")[0] if ":" in label else label
        _label_color[label] = _PARENT_COLORS.get(parent, "#888888")
    return _label_color[label]


def show(idx: int) -> None:
    row   = ds[idx]
    image = row["image"].copy().convert("RGB")
    draw  = ImageDraw.Draw(image)
    w, h  = image.size

    for ann in row["annotations"]:
        color = _color_for(ann["label"])
        for bbox in ann["bboxes"]:
            y_min, x_min, y_max, x_max = bbox
            draw.rectangle(
                [x_min * w, y_min * h, x_max * w, y_max * h],
                outline=color, width=2,
            )

    status = "REJECTED" if idx in rejected else "ok"
    source = row.get("source_pdf", "?")
    page   = row.get("page", "?")
    total  = row.get("total_pages", "?")
    dtype  = row.get("doc_type", "?")
    meta   = f"{source}  p.{page}/{total}  [{dtype}]"
    title  = f"[{idx}/{len(ds)-1}]  {meta}  —  {status}"

    fig, ax = plt.subplots(figsize=(10, 13))
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(title, fontsize=11, color="red" if status == "REJECTED" else "black")

    legend = [mpatches.Patch(color=c, label=l) for l, c in _label_color.items()]
    if legend:
        ax.legend(handles=legend, loc="upper right", fontsize=8,
                  bbox_to_anchor=(1.18, 1), borderaxespad=0)

    plt.tight_layout()
    plt.show()

    print(f"{'ID':<36}  {'LABEL':<30}  TEXT")
    print("-" * 90)
    for ann in row["annotations"]:
        print(f"{ann['id']:<36}  {ann['label']:<30}  {ann['text']}")

In [ ]:
# ── Rejection state ───────────────────────────────────────────────────────────
# Loads existing review file if present so you can resume across sessions.
if REVIEW_PATH.exists():
    rejected: set[int] = set(json.loads(REVIEW_PATH.read_text())["rejected"])
    print(f"Loaded {len(rejected)} previously rejected indices from '{REVIEW_PATH}'")
else:
    rejected: set[int] = set()


def reject(idx: int) -> None:
    rejected.add(idx)
    print(f"Marked {idx} as REJECTED  ({len(rejected)} total)")

def approve(idx: int) -> None:
    rejected.discard(idx)
    print(f"Marked {idx} as OK  ({len(rejected)} rejected total)")

def save() -> None:
    REVIEW_PATH.write_text(json.dumps({"rejected": sorted(rejected)}, indent=2))
    print(f"Saved {len(rejected)} rejected indices to '{REVIEW_PATH}'")

In [ ]:
# ── Navigation ────────────────────────────────────────────────────────────────
# Call next_sample() / prev_sample() / go(n) in the cell below and hit Shift+Enter.
_cursor = [-1]

def next_sample():
    _cursor[0] = min(len(ds) - 1, _cursor[0] + 1)
    show(_cursor[0])

def prev_sample():
    _cursor[0] = max(0, _cursor[0] - 1)
    show(_cursor[0])

def go(idx: int):
    _cursor[0] = idx
    show(_cursor[0])

In [ ]:
# ── Run this cell (Shift+Enter) to navigate and mark ─────────────────────────
next_sample()        # prev_sample() | go(n) | reject(_cursor[0]) | approve(_cursor[0]) | save()